# Associeren met Apriori

We gaan associeren = het zoeken van subgroepen in de dataset die een aantal kenmerken gemeen hebben.

# Voorbeeld 1 Online Retail Data
We importeren eerst de benodigde libraries.

In [ ]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

Vervolgens lezen we de data in.

In [ ]:
data = pd.read_excel('Online Retail.xlsx')
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
data.shape

(541909, 8)

Er zitten 541.909 regels in de dataset en 8 variabelen:

In [ ]:
data.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

We hebben informatie uit 37 landen:

In [ ]:
data.Country.unique()

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Bahrain', 'Greece', 'Hong Kong', 'Singapore',
       'Lebanon', 'United Arab Emirates', 'Saudi Arabia',
       'Czech Republic', 'Canada', 'Unspecified', 'Brazil', 'USA',
       'European Community', 'Malta', 'RSA'], dtype=object)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


Er zitten 541.909 regels in de database. <br>
Niet alle records zijn gevuld, dat wil zeggen dat we de data moeten schonen. <p>



In [ ]:
# Stripping extra spaces in the description
data['Description'] = data['Description'].str.strip()
  
# Dropping the rows without any invoice number
data.dropna(axis = 0, subset =['InvoiceNo'], inplace = True)

# Converting the invoice number to a string
data['InvoiceNo'] = data['InvoiceNo'].astype('str')
  
# Dropping all transactions which were done on credit
data = data[~data['InvoiceNo'].str.contains('C')]

Na de schoning moeten we nu de data herordenen (= consolidate the items into 1 transaction per row based on invoice number) en hercoderen met behulp van hot encode.

In [ ]:
# Transactions done in France
basket_France = (data[data['Country'] =="France"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))
  
# Transactions done in the United Kingdom
basket_UK = (data[data['Country'] =="United Kingdom"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))
  
# Transactions done in The Netherlands
basket_NL = (data[data['Country'] =="Netherlands"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

In [ ]:
# Defining the hot encoding function to make the data suitable 
# for the concerned libraries
def hot_encode(x):
    if(x <= 0):
        return 0
    if(x >= 1):
        return 1
  
# Encoding the datasets
basket_encoded = basket_France.applymap(hot_encode)
basket_France = basket_encoded
  
basket_encoded = basket_UK.applymap(hot_encode)
basket_UK = basket_encoded
  
basket_encoded = basket_NL.applymap(hot_encode)
basket_NL = basket_encoded

En nu eindelijk: associeren met het apriori-algoritme.

In [ ]:
# Building the model
frq_items = apriori(basket_France, min_support = 0.05, use_colnames = True)
  
# Collecting the inferred rules in a dataframe
rules = association_rules(frq_items, metric ="lift", min_threshold = 1)
rules = rules.sort_values(['confidence', 'lift'], ascending =[False, False])
print(rules.head())

                                           antecedents  \
45                        (JUMBO BAG WOODLAND ANIMALS)   
258  (RED TOADSTOOL LED NIGHT LIGHT, PLASTERS IN TI...   
270  (RED TOADSTOOL LED NIGHT LIGHT, PLASTERS IN TI...   
302  (SET/20 RED RETROSPOT PAPER NAPKINS, SET/6 RED...   
300  (SET/6 RED SPOTTY PAPER PLATES, SET/20 RED RET...   

                         consequents  antecedent support  consequent support  \
45                         (POSTAGE)            0.076531            0.765306   
258                        (POSTAGE)            0.051020            0.765306   
270                        (POSTAGE)            0.053571            0.765306   
302  (SET/6 RED SPOTTY PAPER PLATES)            0.102041            0.127551   
300    (SET/6 RED SPOTTY PAPER CUPS)            0.102041            0.137755   

      support  confidence      lift  leverage  conviction  
45   0.076531       1.000  1.306667  0.017961         inf  
258  0.051020       1.000  1.306667  0.011974     

En wat weten we nu?<br>

We kijken naar transactiedata van Frankrijk, en zien dat:<br>
- papieren borden vaak samen met papieren bekers en servetten verkocht worden (regel 301 en 302, lift van 7,0 en 7,6)
- een rode paddestoel als nachtlampje & (dieren)pleisters samen verkocht worden met postzegels (?)

Eens kijken wat er bij het Verenigd Koninkrijk uitkomt:

In [ ]:
# Building the model
frq_items = apriori(basket_UK, min_support = 0.01, use_colnames = True)
  
# Collecting the inferred rules in a dataframe
rules = association_rules(frq_items, metric ="lift", min_threshold = 1)
rules = rules.sort_values(['confidence', 'lift'], ascending =[False, False])
print(rules.head())

                                       antecedents             consequents  \
117           (BEADED CRYSTAL HEART PINK ON STICK)        (DOTCOM POSTAGE)   
2020  (SUKI  SHOULDER BAG, JAM MAKING SET PRINTED)        (DOTCOM POSTAGE)   
2294         (HERB MARKER THYME, HERB MARKER MINT)  (HERB MARKER ROSEMARY)   
2302   (HERB MARKER PARSLEY, HERB MARKER ROSEMARY)     (HERB MARKER THYME)   
2301      (HERB MARKER THYME, HERB MARKER PARSLEY)  (HERB MARKER ROSEMARY)   

      antecedent support  consequent support   support  confidence       lift  \
117             0.011036            0.037928  0.010768    0.975728  25.725872   
2020            0.011625            0.037928  0.011196    0.963134  25.393807   
2294            0.010714            0.012375  0.010232    0.955000  77.173095   
2302            0.011089            0.012321  0.010553    0.951691  77.240055   
2301            0.011089            0.012375  0.010553    0.951691  76.905682   

      leverage  conviction  
117   0.010349 

Kennelijk kopen de Britten:<br>
- een HERB MARKER THYME samen met een HERB MARKER ROSEMARY (lift van 77)<br>
- een schoudertas met JAM MAKING SET PRINT via een website (?)
<p>
<br>
  
Tot slot: de Nederlanders.

In [ ]:
# Building the model
frq_items = apriori(basket_NL, min_support = 0.05, use_colnames = True)
  
# Collecting the inferred rules in a dataframe
rules = association_rules(frq_items, metric ="lift", min_threshold = 1)
rules = rules.sort_values(['confidence', 'lift'], ascending =[False, False])
print(rules.head())

                                            antecedents  \
650                      (FOLDING BUTTERFLY MIRROR RED)   
651                 (FOLDING BUTTERFLY MIRROR HOT PINK)   
1351  (FOOD CONTAINER SET 3 LOVE HEART, CARD DOLLY G...   
1354                           (10 COLOUR SPACEBOY PEN)   
1382  (CARD DOLLY GIRL, STRAWBERRY LUNCH BOX WITH CU...   

                                            consequents  antecedent support  \
650                 (FOLDING BUTTERFLY MIRROR HOT PINK)            0.052632   
651                      (FOLDING BUTTERFLY MIRROR RED)            0.052632   
1351                           (10 COLOUR SPACEBOY PEN)            0.052632   
1354  (FOOD CONTAINER SET 3 LOVE HEART, CARD DOLLY G...            0.052632   
1382                           (10 COLOUR SPACEBOY PEN)            0.052632   

      consequent support   support  confidence  lift  leverage  conviction  
650             0.052632  0.052632         1.0  19.0  0.049861         inf  
651             

Populair bij de Hollanders: de [10 colour Spaceboy pen](https://www.petit-bazaar.com/products/rex-10-colour-spaceboy-pen-stationery-rex-22418?variant=20775452549) met de [Card Dolly Girl inc Lunchbox](https://www.bol.com/nl/nl/p/vintage-dolly-girl-lunchtrommel-en-drinkbeker-retro-lunch-school-set/9200000092066670/)!

# Voorbeeld 2 Pinguins

Apriori-analyse per pinguinsoort:

In [ ]:
basket_Adelie =(df[df['species'] == "Adelie"]
                .groupby(['body_mass_g', 'flipper_length_mm'])['culmen_length_mm']
                .sum().unstack().reset_index().fillna(0)
                .set_index('body_mass_g'))



basket_Chinstrap =(df[df['species'] == "Chinstrap"]
                .groupby(['body_mass_g', 'flipper_length_mm'])['culmen_length_mm']
                .sum().unstack().reset_index().fillna(0)
                .set_index('body_mass_g'))



basket_Gentoo =(df[df['species'] == "Gentoo"]
                .groupby(['body_mass_g', 'flipper_length_mm'])['culmen_length_mm']
                .sum().unstack().reset_index().fillna(0)
                .set_index('body_mass_g'))

In [ ]:
def hot_encode(x):
    if(x <= 0):
        return 0
    if(x >= 1):
        return 1

basket_encoded = basket_Adelie.applymap(hot_encode)
basket_Adelie = basket_encoded

basket_encoded = basket_Chinstrap.applymap(hot_encode)
basket_Chinstrap = basket_encoded

basket_encoded = basket_Gentoo.applymap(hot_encode)
basket_Gentoo = basket_encoded

In [ ]:
frq_items = apriori(basket_Adelie, min_support = 0.05, use_colnames = True)

rules = association_rules(frq_items, metric = "lift", min_threshold = 1)
rules = rules.sort_values(["confidence", "lift"], ascending = [False, False])

rules.head()

In [ ]:
frq_items = apriori(basket_Chinstrap, min_support = 0.05, use_colnames = True)

rules = association_rules(frq_items, metric = "lift", min_threshold = 1)
rules = rules.sort_values(["confidence", "lift"], ascending = [False, False])

rules.head()

In [ ]:
frq_items = apriori(basket_Gentoo, min_support = 0.05, use_colnames = True)

rules = association_rules(frq_items, metric = "lift", min_threshold = 1)
rules = rules.sort_values(["confidence", "lift"], ascending = [False, False])

rules.head()

Conclusie?

# Voorbeeld 3 VU Studentendataset?